# 01 - Data Cleaning, OCR Risalah & Pembagian Dataset 80:20

Pipeline:
1. OCR seluruh PDF risalah → simpan teks ke `dataset/02_extracted/ocr_risalah/`
2. Pasangkan transkripsi Whisper (source) dengan teks OCR (target)
3. Bersihkan & validasi pasangan data
4. Split otomatis 80% latih / 20% uji → `train.csv` & `test.csv`

## 1. Import Library

In [1]:
import pandas as pd
import re
from pathlib import Path
import sys
sys.path.append('..')
from modules.ocr_risalah import proses_semua_pdf

## 2. OCR Semua Risalah PDF

In [2]:
DIR_PDF    = Path('../dataset/01_raw/risalah_pdf')
DIR_OCR    = Path('../dataset/02_extracted/ocr_risalah')

# Sesuaikan pola_awal dan pola_akhir dengan format risalah kamu!
POLA_AWAL  = r'MENYANYIKAN LAGU INDONESIA RAYA'
POLA_AKHIR = r'RAPAT DITUTUP PUKUL'

hasil_ocr = proses_semua_pdf(
    direktori_pdf=DIR_PDF,
    direktori_output=DIR_OCR,
    pola_awal=POLA_AWAL,
    pola_akhir=POLA_AKHIR,
)
print(f'Total risalah berhasil di-OCR: {len(hasil_ocr)}')

✅ Paripurna_Ke_10_Persidangan_II_2024_2025.pdf → 4094 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_II_2024_2025.pdf → 711 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_III_2023_2024.pdf → 2469 kata diekstrak
✅ Paripurna_Ke_12_Persidangan_III_2023_2024.pdf → 3676 kata diekstrak
✅ Paripurna_Ke_13_Persidangan_IV_2023_2024.pdf → 6099 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_II_2024_2025.pdf → 1613 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_IV_2023_2024.pdf → 9899 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_II_2024_2025.pdf → 6006 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_IV_2023_2024.pdf → 2325 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_II_2024_2025.pdf → 2543 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_V_2023_2024.pdf → 3956 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_III_2024_2025.pdf → 852 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_V_2023_2024.pdf → 5023 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_III_2024_2025.pdf → 5208 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_V_20

## 3. Load Transkripsi Whisper & Pasangkan dengan OCR

In [3]:
DIR_TRANSKRIP = Path('../dataset/02_extracted/whisper_transcripts')

baris = []
for txt_file in sorted(DIR_TRANSKRIP.glob('*.txt')):
    nama_tanpa_ext = txt_file.stem
    ocr_file = DIR_OCR / (nama_tanpa_ext + '.txt')
    if not ocr_file.exists():
        print(f'[SKIP] Tidak ada pasangan OCR untuk: {txt_file.name}')
        continue
    source = txt_file.read_text(encoding='utf-8').strip()
    target = ocr_file.read_text(encoding='utf-8').strip()
    if source and target:
        baris.append({'source': source, 'target': target})

df = pd.DataFrame(baris)
print(f'Total pasangan data: {len(df)}')
df.head()

Total pasangan data: 30


,source,target
0,Hadirin sekalian. Rami persilakan untuk duduk ...,"Hadirin sekalian, kami persilakan untuk duduk ..."
1,Hadirin kami persilakan untuk duduk kembali. S...,"Hadirin, kami persilakan untuk duduk kembali.\..."
2,Selanjutnya kepada hadirin sekalian untuk dapa...,Selanjutnya kepada Hadirin sekalian untuk dapa...
3,Hadirin kami bersilakan untuk duduk kembali. S...,Kami persilahkan untuk duduk kembali.\n\nSesua...
4,Hadirin kami persilakan untuk duduk kembali. S...,"Hadirin, kami persilakan untuk duduk Kembali.\..."


## 4. Validasi & Bersihkan

In [4]:
# Hapus baris kosong
df = df.dropna(subset=['source', 'target']).reset_index(drop=True)

# Filter pasangan yang terlalu pendek (mungkin hasil OCR gagal)
df = df[df['source'].str.split().str.len() >= 20].reset_index(drop=True)
df = df[df['target'].str.split().str.len() >= 10].reset_index(drop=True)

print(f'Data valid setelah cleaning: {len(df)} baris')
df[['source', 'target']].apply(lambda c: c.str.split().str.len()).describe()

Data valid setelah cleaning: 30 baris


,source,target
count,30.000000,30.000000
mean,5444.666667,5910.466667
std,3779.044482,4095.431969
min,673.000000,711.000000
25%,2286.000000,2591.750000
50%,4502.500000,4912.000000
75%,8488.250000,9200.000000
max,13011.000000,14464.000000


## 5. Split 80:20 → train.csv & test.csv

> Tidak ada validation set. Data dibagi langsung menjadi **data latih (80%)** dan **data uji (20%)**.

In [6]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Data latih (train): {len(df_train)} ({len(df_train)/len(df)*100:.0f}%)')
print(f'Data uji  (test) : {len(df_test)}  ({len(df_test)/len(df)*100:.0f}%)')

Data latih (train): 24 (80%)
Data uji  (test) : 6  (20%)


## 6. Simpan ke CSV

In [7]:
DATA_DIR = Path('../dataset/03_paired')
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(DATA_DIR / 'train.csv', index=False, encoding='utf-8')
df_test.to_csv(DATA_DIR  / 'test.csv',  index=False, encoding='utf-8')

print('Dataset berhasil disimpan:')
print(f'  train.csv → {len(df_train)} baris')
print(f'  test.csv  → {len(df_test)} baris')

Dataset berhasil disimpan:
  train.csv → 24 baris
  test.csv  → 6 baris
